# Job Listing Parser

An AI-powered tool that extracts structured data from raw job postings using **Pydantic** and **OpenAI Structured Output**.

**Concepts demonstrated:**
- Pydantic BaseModel for data validation and type safety
- OpenAI `beta.chat.completions.parse()` for structured output
- Nested Pydantic models for complex data structures
- Batch processing of multiple job listings
- Comparison and analysis across parsed listings

## 1. Setup

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from IPython.display import display, Markdown

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)

print("Setup complete.")

Setup complete.


## 2. Pydantic Basics - Why Structured Output Matters

Without Pydantic, LLM responses are just raw text. Pydantic lets us:
- **Validate** that the response has the right fields and types
- **Parse** JSON into Python objects with autocomplete support
- **Catch errors** when the LLM returns unexpected data

In [2]:
# Simple example: a Pydantic model for a job skill
class Skill(BaseModel):
    name: str
    level: str  # "required" or "nice-to-have"

# Valid input - works fine
skill = Skill(name="Python", level="required")
print(f"Valid: {skill.model_dump_json()}")

# Invalid input - Pydantic catches the error
try:
    bad_skill = Skill(name=123, level="required")  # name should be str
    print(f"Accepted: {bad_skill.model_dump_json()}")  # Pydantic coerces 123 to "123"
except Exception as e:
    print(f"Error: {e}")

Valid: {"name":"Python","level":"required"}
Error: 1 validation error for Skill
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


## 3. Define the Job Listing Data Model

We define nested Pydantic models that describe the structure we want the LLM to extract from raw job postings.

In [3]:
from typing import Optional


class SalaryInfo(BaseModel):
    min_salary: Optional[int] = None
    max_salary: Optional[int] = None
    currency: str = "USD"
    period: str = "yearly"  # yearly, monthly, hourly


class SkillRequirement(BaseModel):
    name: str
    category: str  # "technical", "soft", "tool", "language"
    importance: str  # "required", "preferred", "nice-to-have"


class JobListing(BaseModel):
    title: str
    company: str
    location: str
    remote_policy: str  # "remote", "hybrid", "on-site", "not specified"
    experience_years: Optional[str] = None  # e.g. "3-5" or "5+"
    seniority_level: str  # "entry", "mid", "senior", "lead", "director"
    salary: Optional[SalaryInfo] = None
    skills: list[SkillRequirement]
    responsibilities: list[str]
    benefits: list[str]
    education: Optional[str] = None
    summary: str  # 1-2 sentence summary of the role


print("Models defined:")
print(f"  - SalaryInfo: {list(SalaryInfo.model_fields.keys())}")
print(f"  - SkillRequirement: {list(SkillRequirement.model_fields.keys())}")
print(f"  - JobListing: {list(JobListing.model_fields.keys())}")

Models defined:
  - SalaryInfo: ['min_salary', 'max_salary', 'currency', 'period']
  - SkillRequirement: ['name', 'category', 'importance']
  - JobListing: ['title', 'company', 'location', 'remote_policy', 'experience_years', 'seniority_level', 'salary', 'skills', 'responsibilities', 'benefits', 'education', 'summary']


## 4. Sample Job Postings

Let's define a few realistic job postings in raw text format - the kind you'd copy-paste from LinkedIn or Indeed.

In [4]:
job_posting_1 = """
Data Engineer - TechFlow Solutions
Location: Milan, Italy (Hybrid - 3 days in office)

About Us:
TechFlow Solutions is a fast-growing data consultancy helping enterprises modernize 
their data infrastructure. We work with clients across finance, retail, and healthcare.

The Role:
We're looking for a Data Engineer with 3-5 years of experience to design and maintain 
our data pipelines. You'll work closely with data scientists and analysts to ensure 
reliable, scalable data delivery.

Responsibilities:
- Design and build ETL/ELT pipelines using Python and SQL
- Manage and optimize data warehouses (Snowflake, BigQuery)
- Implement data quality checks and monitoring
- Collaborate with stakeholders to understand data requirements
- Maintain documentation and data catalogs
- Participate in on-call rotation for data platform issues

Requirements:
- 3-5 years of experience in data engineering
- Strong Python and SQL skills
- Experience with cloud platforms (AWS, GCP, or Azure)
- Knowledge of Apache Spark or similar distributed processing frameworks
- Familiarity with orchestration tools (Airflow, Dagster, or Prefect)
- Good communication skills and ability to work in a team
- Bachelor's degree in Computer Science, Engineering, or related field

Nice to Have:
- Experience with dbt (data build tool)
- Knowledge of Kafka or other streaming platforms
- Terraform or other IaC tools

Salary: EUR 45,000 - 60,000 per year
Benefits: Remote work flexibility, learning budget (1500 EUR/year), health insurance, 
team retreats, flexible hours, stock options after 1 year
"""

job_posting_2 = """
Frontend Developer
Bright Pixel Studio - Rome, Italy
Full Remote Available

We're a creative digital agency building beautiful web experiences for fashion 
and lifestyle brands. Our small team of 12 is passionate about design and performance.

What you'll do:
- Build responsive, pixel-perfect UIs from Figma designs
- Develop and maintain React/Next.js applications
- Optimize web performance (Core Web Vitals)
- Write unit and integration tests
- Review code and mentor junior developers
- Work directly with designers and clients

What we need:
- 2+ years with React and TypeScript
- Solid CSS skills (Tailwind CSS preferred)
- Experience with Next.js and server-side rendering
- Understanding of accessibility (WCAG) standards
- Good eye for design and attention to detail
- English fluency (our team is international)

Bonus points:
- Experience with animation libraries (Framer Motion, GSAP)
- Three.js or WebGL knowledge
- Contributions to open source

We offer:
- Competitive salary based on experience
- Fully remote option
- 4-day work week (Fridays off!)
- Latest MacBook Pro
- Conference budget
- Annual team trip
"""

job_posting_3 = """
AI/ML Engineer - Senior
NovaMind AI | San Francisco, CA (On-site)

NovaMind AI is building the next generation of AI-powered healthcare diagnostics. 
Backed by $25M Series A, we're scaling our engineering team.

About the Role:
As a Senior ML Engineer, you'll lead the development of our core diagnostic models, 
working at the intersection of deep learning and medical imaging. This is a high-impact 
role where your work directly improves patient outcomes.

Key Responsibilities:
- Design and train deep learning models for medical image analysis
- Build and maintain ML pipelines (training, evaluation, deployment)
- Collaborate with medical professionals to validate model performance
- Optimize model inference for real-time clinical use
- Publish research and present at conferences
- Mentor junior ML engineers

Qualifications:
- MS or PhD in Computer Science, Machine Learning, or related field
- 5+ years of experience in machine learning engineering
- Deep expertise in PyTorch and computer vision
- Experience with medical imaging (DICOM, radiology) is a strong plus
- Published research in top ML conferences (NeurIPS, ICML, CVPR)
- Strong Python and software engineering practices
- Experience with MLOps (MLflow, Weights & Biases, Kubeflow)
- Excellent problem-solving and communication skills

Compensation:
- Base salary: $180,000 - $250,000
- Equity: 0.1% - 0.3%
- Health, dental, vision insurance
- Unlimited PTO
- $5,000 annual learning stipend
- Relocation assistance available
"""

all_postings = {
    "Data Engineer": job_posting_1,
    "Frontend Developer": job_posting_2,
    "AI/ML Engineer": job_posting_3
}

print(f"Loaded {len(all_postings)} job postings")
for name in all_postings:
    print(f"  - {name}")

Loaded 3 job postings
  - Data Engineer
  - Frontend Developer
  - AI/ML Engineer


## 5. Parse a Single Job Listing with OpenAI Structured Output

We use `beta.chat.completions.parse()` with our Pydantic model as `response_format`.
OpenAI guarantees the response matches our schema - no manual JSON parsing needed!

In [5]:
def parse_job_listing(raw_text: str) -> JobListing:
    """Parse a raw job posting into a structured JobListing using OpenAI."""
    prompt = f"""Extract structured information from this job posting.
Analyze carefully and fill in all fields. For skills, categorize each as:
- category: "technical", "soft", "tool", or "language"
- importance: "required", "preferred", or "nice-to-have"

If salary is not explicitly mentioned, set salary to null.
For remote_policy, choose: "remote", "hybrid", "on-site", or "not specified".
For seniority_level, choose: "entry", "mid", "senior", "lead", or "director".

Job Posting:
{raw_text}"""

    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format=JobListing
    )

    return response.choices[0].message.parsed


print("parse_job_listing() defined.")

parse_job_listing() defined.


In [6]:
# Parse the first job posting
parsed_job_1 = parse_job_listing(job_posting_1)

print(f"Title: {parsed_job_1.title}")
print(f"Company: {parsed_job_1.company}")
print(f"Location: {parsed_job_1.location}")
print(f"Remote: {parsed_job_1.remote_policy}")
print(f"Seniority: {parsed_job_1.seniority_level}")
print(f"Experience: {parsed_job_1.experience_years}")

if parsed_job_1.salary:
    print(f"Salary: {parsed_job_1.salary.currency} {parsed_job_1.salary.min_salary:,} - {parsed_job_1.salary.max_salary:,} ({parsed_job_1.salary.period})")

print(f"\nSkills ({len(parsed_job_1.skills)}):")
for s in parsed_job_1.skills:
    print(f"  [{s.importance}] {s.name} ({s.category})")

print(f"\nBenefits: {', '.join(parsed_job_1.benefits)}")
print(f"\nSummary: {parsed_job_1.summary}")

Title: Data Engineer
Company: TechFlow Solutions
Location: Milan, Italy
Remote: hybrid
Seniority: mid
Experience: 3-5 years
Salary: EUR 45,000 - 60,000 (per year)

Skills (14):
  [required] Python (language)
  [required] SQL (language)
  [required] AWS (technical)
  [required] GCP (technical)
  [required] Azure (technical)
  [required] Apache Spark (technical)
  [required] Airflow (tool)
  [required] Dagster (tool)
  [required] Prefect (tool)
  [required] Communication skills (soft)
  [required] Teamwork (soft)
  [nice-to-have] dbt (data build tool) (tool)
  [nice-to-have] Kafka (technical)
  [nice-to-have] Terraform (tool)

Benefits: Remote work flexibility, Learning budget (1500 EUR/year), Health insurance, Team retreats, Flexible hours, Stock options after 1 year

Summary: TechFlow Solutions is seeking a Data Engineer to design and maintain data pipelines, working closely with data scientists and analysts to ensure reliable, scalable data delivery. The role requires 3-5 years of exp

In [7]:
# See the full structured JSON
print(json.dumps(parsed_job_1.model_dump(), indent=2))

{
  "title": "Data Engineer",
  "company": "TechFlow Solutions",
  "location": "Milan, Italy",
  "remote_policy": "hybrid",
  "experience_years": "3-5 years",
  "seniority_level": "mid",
  "salary": {
    "min_salary": 45000,
    "max_salary": 60000,
    "currency": "EUR",
    "period": "per year"
  },
  "skills": [
    {
      "name": "Python",
      "category": "language",
      "importance": "required"
    },
    {
      "name": "SQL",
      "category": "language",
      "importance": "required"
    },
    {
      "name": "AWS",
      "category": "technical",
      "importance": "required"
    },
    {
      "name": "GCP",
      "category": "technical",
      "importance": "required"
    },
    {
      "name": "Azure",
      "category": "technical",
      "importance": "required"
    },
    {
      "name": "Apache Spark",
      "category": "technical",
      "importance": "required"
    },
    {
      "name": "Airflow",
      "category": "tool",
      "importance": "required"
    },

## 6. Batch Parse All Job Listings

Now let's parse all three postings and compare them side by side.

In [8]:
# Parse all postings
parsed_jobs = {}
for name, text in all_postings.items():
    print(f"Parsing: {name}...")
    parsed_jobs[name] = parse_job_listing(text)

print(f"\nAll {len(parsed_jobs)} jobs parsed successfully!")

Parsing: Data Engineer...
Parsing: Frontend Developer...
Parsing: AI/ML Engineer...

All 3 jobs parsed successfully!


In [9]:
# Comparison table
def display_comparison(jobs: dict[str, JobListing]):
    """Display a markdown comparison table of parsed jobs."""
    rows = []
    rows.append("| Field | " + " | ".join(jobs.keys()) + " |")
    rows.append("|" + "---|" * (len(jobs) + 1))

    # Title
    rows.append("| **Title** | " + " | ".join(j.title for j in jobs.values()) + " |")
    # Company
    rows.append("| **Company** | " + " | ".join(j.company for j in jobs.values()) + " |")
    # Location
    rows.append("| **Location** | " + " | ".join(j.location for j in jobs.values()) + " |")
    # Remote
    rows.append("| **Remote** | " + " | ".join(j.remote_policy for j in jobs.values()) + " |")
    # Seniority
    rows.append("| **Seniority** | " + " | ".join(j.seniority_level for j in jobs.values()) + " |")
    # Experience
    rows.append("| **Experience** | " + " | ".join(j.experience_years or 'N/A' for j in jobs.values()) + " |")
    # Salary
    salaries = []
    for j in jobs.values():
        if j.salary and j.salary.min_salary:
            salaries.append(f"{j.salary.currency} {j.salary.min_salary:,}-{j.salary.max_salary:,}")
        else:
            salaries.append("Not specified")
    rows.append("| **Salary** | " + " | ".join(salaries) + " |")
    # Skills count
    rows.append("| **Skills** | " + " | ".join(str(len(j.skills)) + " listed" for j in jobs.values()) + " |")
    # Benefits count
    rows.append("| **Benefits** | " + " | ".join(str(len(j.benefits)) + " listed" for j in jobs.values()) + " |")

    display(Markdown("\n".join(rows)))


display_comparison(parsed_jobs)

| Field | Data Engineer | Frontend Developer | AI/ML Engineer |
|---|---|---|---|
| **Title** | Data Engineer | Frontend Developer | AI/ML Engineer - Senior |
| **Company** | TechFlow Solutions | Bright Pixel Studio | NovaMind AI |
| **Location** | Milan, Italy | Rome, Italy | San Francisco, CA |
| **Remote** | hybrid | remote | on-site |
| **Seniority** | mid | mid | senior |
| **Experience** | 3-5 years | 2+ years | 5+ years |
| **Salary** | EUR 45,000-60,000 | Not specified | USD 180,000-250,000 |
| **Skills** | 14 listed | 12 listed | 8 listed |
| **Benefits** | 6 listed | 6 listed | 6 listed |

## 7. Skills Analysis Across Jobs

Let's use another Pydantic model to get a structured skills analysis from the LLM.

In [11]:
class RoleSkills(BaseModel):
    role_name: str
    unique_skills: list[str]


class SkillsAnalysis(BaseModel):
    common_skills: list[str]
    unique_to_each: list[RoleSkills]
    most_in_demand_category: str
    career_advice: str


# Collect all skills from parsed jobs
skills_summary = ""
for name, job in parsed_jobs.items():
    skills_list = ", ".join(f"{s.name} ({s.importance})" for s in job.skills)
    skills_summary += f"\n{name} ({job.title}):\n{skills_list}\n"

prompt = f"""Analyze the skills across these job listings and provide insights.

{skills_summary}

Identify:
- Skills that appear in multiple listings (common_skills)
- Skills unique to each role (unique_to_each as a list, one entry per role)
- The most in-demand skill category overall
- Brief career advice for someone looking at these roles"""

response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    response_format=SkillsAnalysis
)

analysis = response.choices[0].message.parsed

print("=== Skills Analysis ===")
print(f"\nCommon skills: {', '.join(analysis.common_skills)}")
print(f"\nMost in-demand category: {analysis.most_in_demand_category}")
print(f"\nUnique skills per role:")
for role in analysis.unique_to_each:
    print(f"  {role.role_name}: {', '.join(role.unique_skills)}")
print(f"\nCareer advice: {analysis.career_advice}")

=== Skills Analysis ===

Common skills: Python, Communication skills

Most in-demand category: Programming Languages

Unique skills per role:
  Data Engineer: SQL, AWS, GCP, Azure, Apache Spark, Airflow, Dagster, Prefect, Teamwork, dbt (data build tool), Kafka, Terraform
  Frontend Developer: React, TypeScript, CSS, Tailwind CSS, Next.js, Server-side rendering, Accessibility (WCAG) standards, Design and attention to detail, English fluency, Animation libraries (Framer Motion, GSAP), Three.js or WebGL, Open source contributions
  AI/ML Engineer: Deep Learning, Medical Imaging (DICOM, radiology), PyTorch, Computer Vision, MLOps (MLflow, Weights & Biases, Kubeflow), Problem-solving

Career advice: To excel in these roles, focus on mastering Python, as it is a common requirement across multiple positions. For Data Engineers, gaining proficiency in cloud platforms like AWS, GCP, and Azure, along with tools like Apache Spark and Airflow, will be beneficial. Frontend Developers should priorit

## 8. Job Match Scorer

Given a candidate's skills, let's score how well they match each job listing.
This uses another Pydantic model for structured scoring output.

In [12]:
class JobMatch(BaseModel):
    job_title: str
    match_score: int  # 0-100
    matching_skills: list[str]
    missing_skills: list[str]
    recommendation: str


class CandidateAnalysis(BaseModel):
    matches: list[JobMatch]
    best_fit: str
    learning_roadmap: list[str]


# Define a sample candidate profile
candidate_skills = """Python, SQL, JavaScript, React, pandas, scikit-learn, 
Git, Docker, AWS, basic Spark knowledge, good communication skills, 
English fluent, Italian native, Bachelor's in Computer Science"""

# Build the job summaries
jobs_text = ""
for name, job in parsed_jobs.items():
    skills_list = ", ".join(f"{s.name} ({s.importance})" for s in job.skills)
    jobs_text += f"\n{job.title} at {job.company}:\nSkills: {skills_list}\nExperience: {job.experience_years}\n"

prompt = f"""Analyze how well this candidate matches each job listing.
Score each job 0-100 based on skill match, and provide actionable feedback.

Candidate Profile:
{candidate_skills}

Job Listings:
{jobs_text}

For learning_roadmap, suggest 3-5 specific skills the candidate should learn next."""

response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    response_format=CandidateAnalysis
)

candidate_analysis = response.choices[0].message.parsed

print("=== Job Match Results ===")
for match in candidate_analysis.matches:
    print(f"\n{match.job_title}: {match.match_score}/100")
    print(f"  Matching: {', '.join(match.matching_skills)}")
    print(f"  Missing: {', '.join(match.missing_skills)}")
    print(f"  Tip: {match.recommendation}")

print(f"\nBest fit: {candidate_analysis.best_fit}")
print(f"\nLearning roadmap:")
for i, skill in enumerate(candidate_analysis.learning_roadmap, 1):
    print(f"  {i}. {skill}")

=== Job Match Results ===

Data Engineer at TechFlow Solutions: 60/100
  Matching: Python, SQL, AWS, Communication skills
  Missing: GCP, Azure, Apache Spark, Airflow, Dagster, Prefect, dbt, Kafka, Terraform
  Tip: The candidate has a strong foundation in Python, SQL, and AWS, which are crucial for this role. To improve their match score, they should focus on gaining experience with cloud platforms like GCP and Azure, as well as data orchestration tools such as Airflow and Dagster. Additionally, enhancing their knowledge of Apache Spark and learning about dbt and Kafka would be beneficial.

Frontend Developer at Bright Pixel Studio: 50/100
  Matching: React, English fluency
  Missing: TypeScript, CSS, Tailwind CSS, Next.js, Server-side rendering, Accessibility (WCAG) standards, Design and attention to detail, Animation libraries, Three.js or WebGL, Open source contributions
  Tip: The candidate's experience with React and English fluency are valuable for this position. To increase thei

## 9. Gradio Interface

A user-friendly interface where you can paste any job posting and get structured data back,
plus optionally enter your skills for a match score.

In [13]:
import gradio as gr


def parse_and_analyze(raw_posting: str, candidate_skills_input: str) -> tuple[str, str]:
    """Parse a job posting and optionally score against candidate skills."""
    if not raw_posting or raw_posting.strip() == "":
        return "Please paste a job posting.", ""

    # Parse the job listing
    parsed = parse_job_listing(raw_posting)

    # Format the structured output
    output = f"## {parsed.title}\n"
    output += f"**Company:** {parsed.company}\n\n"
    output += f"**Location:** {parsed.location} ({parsed.remote_policy})\n\n"
    output += f"**Seniority:** {parsed.seniority_level} | **Experience:** {parsed.experience_years or 'N/A'}\n\n"

    if parsed.salary and parsed.salary.min_salary:
        output += f"**Salary:** {parsed.salary.currency} {parsed.salary.min_salary:,} - {parsed.salary.max_salary:,} ({parsed.salary.period})\n\n"

    output += f"**Education:** {parsed.education or 'Not specified'}\n\n"
    output += "### Skills\n"
    for s in parsed.skills:
        icon = {"required": "*", "preferred": "+", "nice-to-have": "-"}.get(s.importance, "-")
        output += f"- {icon} **{s.name}** ({s.category}, {s.importance})\n"

    output += "\n### Key Responsibilities\n"
    for r in parsed.responsibilities[:5]:
        output += f"- {r}\n"

    output += "\n### Benefits\n"
    for b in parsed.benefits:
        output += f"- {b}\n"

    output += f"\n---\n*{parsed.summary}*"

    # Match score if candidate skills provided
    match_output = ""
    if candidate_skills_input and candidate_skills_input.strip():
        skills_text = ", ".join(f"{s.name} ({s.importance})" for s in parsed.skills)

        match_prompt = f"""Score how well this candidate matches the job.

Candidate skills: {candidate_skills_input}

Job: {parsed.title} at {parsed.company}
Required skills: {skills_text}
Experience needed: {parsed.experience_years}

Provide a match score, matching/missing skills, and a recommendation."""

        match_response = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=[{"role": "user", "content": match_prompt}],
            temperature=0,
            response_format=JobMatch
        )
        match = match_response.choices[0].message.parsed

        match_output = f"## Match Score: {match.match_score}/100\n\n"
        match_output += f"**Matching skills:** {', '.join(match.matching_skills)}\n\n"
        match_output += f"**Missing skills:** {', '.join(match.missing_skills)}\n\n"
        match_output += f"**Recommendation:** {match.recommendation}"

    return output, match_output


demo = gr.Interface(
    fn=parse_and_analyze,
    inputs=[
        gr.Textbox(
            lines=12,
            placeholder="Paste a job posting here...",
            label="Job Posting (raw text)"
        ),
        gr.Textbox(
            lines=3,
            placeholder="(Optional) Python, SQL, React, Docker, 3 years experience...",
            label="Your Skills (for match scoring)"
        )
    ],
    outputs=[
        gr.Markdown(label="Parsed Job Listing"),
        gr.Markdown(label="Match Score")
    ],
    title="Job Listing Parser",
    description="Paste any job posting and get structured data extracted using AI + Pydantic. Optionally enter your skills for a match score.",
    flagging_mode="never"
)

demo.launch()

/Users/eugenionex/Downloads/LLM and Agentic AI Bootcamp Materials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
